In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "archive/src/experiment.py").is_file()
)
PROJECT_ROOT = ROOT = REPOSITORY_ROOT / "archive"
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))


# История training strategies ResNet50

Smoothing, CosFace и mining не превзошли stronger50; этот ноутбук сохраняет сравнение завершённых runs. RUN_TRAINING=False по умолчанию.

Актуальные дешёвые эксперименты с сохранёнными probabilities и frozen embeddings — в 08_frozen_pairwise_experiments.ipynb. Финальный train+val refit — отдельно в 09_final_refit.ipynb. Дополнительное обучение full больше не предлагается.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt


from archive.src.experiment import load_config, run_experiment
from archive.src.transforms import preview_augmentations
from archive.src.strategy_analysis import compare_resnet50_runs
from archive.src.utils import get_device


In [ ]:
RUN_TRAINING = False  # анализ имеющихся артефактов; True только для нового обучения
CONFIG = "configs/resnet50_partial_domain_shift_stronger.json"
# CONFIG = "configs/resnet50_partial_domain_shift_smoothing.json"  # уже оценён
# CONFIG = "configs/resnet50_partial_domain_shift_rrc.json"  # уже оценён, хуже базы
# CONFIG = "configs/resnet50_partial_rrc_smoothing.json"
# CONFIG = "configs/resnet50_partial_rrc_cutmix.json"
# CONFIG = "configs/resnet50_partial_domain_shift_cosface.json"
# CONFIG = "configs/resnet50_partial_domain_shift_hard_mining.json"
config = load_config(CONFIG, project_root=PROJECT_ROOT)
EXP_ID = config["exp_id"]  # Если ID уже запускался, задайте новый.
config["training"]["device"] = "auto"
# config["data"]["raw_dir"] = "/kaggle/input/<dataset>"
raw = Path(config["data"].get("raw_dir", "data/raw"))
if not raw.is_absolute():
    raw = PROJECT_ROOT / raw
BASELINE = "resnet50_partial_domain_shift_stronger_001"
print(CONFIG, EXP_ID, get_device(config["training"]["device"]))
print(config["notes"])


## Проверка train-преобразований до дорогого запуска
Галерея показывает только операции над отдельными изображениями. CutMix применяется к батчам; его p/alpha/max_area задаются в training.batch_augmentation. Crop может вырезать часть объекта — проверьте особенно court/church/railway_station. Scale=[0.65,1] означает долю площади, не длины стороны. Val всегда видит полный кадр256.


In [ ]:
raw = Path(config["data"].get("raw_dir", "data/raw"))
if not raw.is_absolute():
    raw = PROJECT_ROOT / raw
train = pd.read_csv(raw / "train.csv", dtype={"id": str})
classes = pd.read_csv(raw / "classes.csv")
for i, name in enumerate(["church", "basketball_court", "railway_station"]):
    matching = classes.loc[classes.class_name == name, "label"]
    if matching.empty:
        continue
    image_id = train.loc[train.label == matching.iloc[0], "id"].iloc[0]
    with Image.open(raw / "train" / image_id) as source:
        fig = preview_augmentations(source.convert("RGB"), config["data"], count=4, seed=42+i)
    fig.suptitle(f"{name}: {image_id}")
    plt.show()
    plt.close(fig)


In [ ]:
if RUN_TRAINING:
    result = run_experiment(config, exp_id=EXP_ID, project_root=PROJECT_ROOT, predict_test=False)
else:
    print("Обучение выключено; ниже можно сравнить сохранённые результаты.")


## Сравнение сохранённых результатов
Эта часть не запускает обучение или inference. На Kaggle перенесите logs/config.json и predictions/val_probabilities.npz базовой модели вместе с train/val/classes CSV. Для smoothing и CutMix обязательно включите также результат RRC: их изолированная база — RRC, а не только старый stronger50.

Accuracy/NLL рассчитаны на одной исходной val. Train loss со smoothing/CosFace или CutMix не равен стандартному CE-loss и напрямую между рецептами не сравнивается; val loss всегда обычный CE. CutMix train Accuracy взвешена площадью смешанных targets.


In [ ]:
# После переноса результатов с Kaggle можно выполнить эту ячейку без ячейки обучения.
CANDIDATES = ["resnet50_partial_domain_shift_rrc_001",
    "resnet50_partial_domain_shift_smoothing_001",
    "resnet50_partial_domain_shift_cosface_001",
    "resnet50_partial_domain_shift_hard_mining_001", EXP_ID]
# CANDIDATES += ["resnet50_partial_domain_shift_rrc_001", "resnet50_partial_rrc_smoothing_001"]
available = [run_id for run_id in CANDIDATES
             if (PROJECT_ROOT / "artifacts/predictions" / run_id / "val_probabilities.npz").exists()]
if not available:
    print("Новые результаты ещё отсутствуют; выполните выбранный запуск на Kaggle.")
elif not (PROJECT_ROOT / "artifacts/predictions" / BASELINE / "val_probabilities.npz").exists():
    print("Перенесите сохранённые probabilities/config baseline stronger50 для сравнения.")
else:
    analysis = compare_resnet50_runs(available, project_root=PROJECT_ROOT, baseline=BASELINE, data_dir=raw)
    display(analysis.comparison)
    display(analysis.per_class.pivot(index="class_name", columns="exp_id", values="errors"))
    display(analysis.changed_cases)
    # Для прямого RRC → smoothing/CutMix сравнения после обоих запусков:
    # pair = compare_resnet50_runs(available, project_root=PROJECT_ROOT,
    #                             baseline="resnet50_partial_domain_shift_rrc_001", data_dir=raw)
    # display(pair.comparison)
